In [1]:
"""
SHAP Analysis for LightGBM v4 Per-Horizon Models
=================================================
Produces:
  1. Mean |SHAP| bar plots per horizon (global feature importance)
  2. SHAP beeswarm plots per horizon (feature value → SHAP direction)
  3. SHAP dependence plots for top-N features per horizon
  4. Combined importance heatmap across all horizons
  5. Feature selection recommendations (drop candidates)

Usage:
  - Adjust DATA_DIR, MODEL_DIR, OUTPUT_DIR paths as needed
  - Run end-to-end; outputs saved as PNGs + a CSV summary

Requirements:
  pip install shap lightgbm pandas numpy matplotlib
"""

import lightgbm as lgb
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import gc
import os
import warnings
warnings.filterwarnings('ignore')


/home/lingod/miniconda3/envs/Hedge_Fund_Time_Series/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =====================================================================
# CONFIG — adjust these paths to match your environment
# =====================================================================
DATA_DIR   = "/home/lingod/kaggle_projects/ts_forecasting/data"
MODEL_DIR  = "/home/lingod/kaggle_projects/ts_forecasting/models"
OUTPUT_DIR = "/home/lingod/kaggle_projects/ts_forecasting/shap_output"

HORIZONS       = [1, 3, 10, 25]
ZERO_CODES     = ['MLAAMU3K', 'EP12UF2K', '1HEMHZK2', 'SJZP0OVU', '83EG83KQ']
SPLIT_INDEX    = 3500
TOP_N_FEATURES = 15       # top features for dependence plots
SHAP_SAMPLE    = 5000     # rows per horizon for SHAP (controls speed vs accuracy)
RANDOM_STATE   = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
# =====================================================================
# 1. LOAD DATA & MODELS
# =====================================================================
print("=" * 70)
print("1. LOADING DATA & MODELS")
print("=" * 70)

df = pd.read_parquet(f"{DATA_DIR}/train.parquet")

cat_features = ['code', 'sub_category', 'horizon']
features = [feat for feat in df.columns
            if feat not in ['id', 'sub_code', 'ts_index', 'weight', 'y_target']]

for feat in cat_features:
    df[feat] = df[feat].astype('category')

val_df = df[df['ts_index'] > SPLIT_INDEX].copy()
del df
gc.collect()
print(f"  Val set size: {len(val_df):,}")
print(f"  Features: {len(features)}")

# Load saved models
models = {}
for h in HORIZONS:
    model_path = f"{MODEL_DIR}/lgb_model_v4_horizon{h}.txt"
    models[h] = lgb.Booster(model_file=model_path)
    print(f"  Loaded model for horizon={h} from {model_path}")


1. LOADING DATA & MODELS
  Val set size: 165,262
  Features: 89
  Loaded model for horizon=1 from /home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_v4_horizon1.txt
  Loaded model for horizon=3 from /home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_v4_horizon3.txt
  Loaded model for horizon=10 from /home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_v4_horizon10.txt
  Loaded model for horizon=25 from /home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_v4_horizon25.txt


In [4]:
# =====================================================================
# 2. COMPUTE SHAP VALUES PER HORIZON
# =====================================================================
print("\n" + "=" * 70)
print("2. COMPUTING SHAP VALUES (TreeExplainer)")
print("=" * 70)

shap_dict = {}       # horizon -> shap_values array (n_samples, n_features)
val_data_dict = {}   # horizon -> DataFrame of sampled val features

for h in HORIZONS:
    print(f"\n  --- Horizon = {h} ---")
    h_val = val_df[val_df['horizon'] == h].copy()

    # Exclude zero-coded rows: their predictions are forced to 0,
    # so SHAP on them is meaningless for model understanding
    h_val = h_val[~h_val['code'].isin(ZERO_CODES)]
    print(f"  Val rows (excl zero-codes): {len(h_val):,}")

    # Sample for speed — SHAP on full val can be slow
    if len(h_val) > SHAP_SAMPLE:
        h_val_sample = h_val.sample(n=SHAP_SAMPLE, random_state=RANDOM_STATE)
        print(f"  Sampled {SHAP_SAMPLE} rows for SHAP computation")
    else:
        h_val_sample = h_val
        print(f"  Using all {len(h_val_sample)} rows")

    X_sample = h_val_sample[features]
    val_data_dict[h] = X_sample

    # TreeExplainer is exact and fast for tree models
    explainer = shap.TreeExplainer(models[h])
    shap_values = explainer.shap_values(X_sample)
    shap_dict[h] = shap_values
    print(f"  SHAP values shape: {shap_values.shape}")

    del h_val, h_val_sample, explainer
    gc.collect()



2. COMPUTING SHAP VALUES (TreeExplainer)

  --- Horizon = 1 ---
  Val rows (excl zero-codes): 35,785
  Sampled 5000 rows for SHAP computation
  SHAP values shape: (5000, 89)

  --- Horizon = 3 ---
  Val rows (excl zero-codes): 35,405
  Sampled 5000 rows for SHAP computation
  SHAP values shape: (5000, 89)

  --- Horizon = 10 ---
  Val rows (excl zero-codes): 33,729
  Sampled 5000 rows for SHAP computation
  SHAP values shape: (5000, 89)

  --- Horizon = 25 ---
  Val rows (excl zero-codes): 31,203
  Sampled 5000 rows for SHAP computation
  SHAP values shape: (5000, 89)


In [5]:
# =====================================================================
# 3. MEAN |SHAP| BAR PLOTS PER HORIZON
# =====================================================================
print("\n" + "=" * 70)
print("3. MEAN |SHAP| BAR PLOTS")
print("=" * 70)

importance_dfs = {}

fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle("Mean |SHAP| per Feature (Top 20) — Per Horizon",
             fontsize=16, fontweight='bold')

for ax, h in zip(axes.flat, HORIZONS):
    sv = shap_dict[h]
    mean_abs_shap = np.abs(sv).mean(axis=0)

    imp_df = pd.DataFrame({
        'feature': features,
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False)
    importance_dfs[h] = imp_df

    top20 = imp_df.head(20)
    ax.barh(top20['feature'][::-1], top20['mean_abs_shap'][::-1],
            color='#3498db', alpha=0.85)
    ax.set_title(f'Horizon = {h}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.96])
save_path = f"{OUTPUT_DIR}/mean_abs_shap_per_horizon.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved: {save_path}")



3. MEAN |SHAP| BAR PLOTS
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/mean_abs_shap_per_horizon.png


In [6]:

# =====================================================================
# 4. SHAP BEESWARM PLOTS PER HORIZON
# =====================================================================
print("\n" + "=" * 70)
print("4. SHAP BEESWARM PLOTS")
print("=" * 70)

for h in HORIZONS:
    fig = plt.figure(figsize=(12, 10))
    shap.summary_plot(
        shap_dict[h],
        val_data_dict[h],
        feature_names=features,
        max_display=20,
        show=False,
        plot_size=None
    )
    plt.title(f'SHAP Beeswarm — Horizon = {h}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_path = f"{OUTPUT_DIR}/shap_beeswarm_h{h}.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")



4. SHAP BEESWARM PLOTS
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_beeswarm_h1.png
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_beeswarm_h3.png
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_beeswarm_h10.png
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_beeswarm_h25.png


In [7]:
# =====================================================================
# 5. SHAP DEPENDENCE PLOTS FOR TOP-N FEATURES
# =====================================================================
print("\n" + "=" * 70)
print(f"5. SHAP DEPENDENCE PLOTS (Top {TOP_N_FEATURES} features per horizon)")
print("=" * 70)

for h in HORIZONS:
    top_features = importance_dfs[h].head(TOP_N_FEATURES)['feature'].tolist()

    n_cols = 3
    n_rows = (TOP_N_FEATURES + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
    fig.suptitle(f'SHAP Dependence Plots — Horizon = {h} (Top {TOP_N_FEATURES})',
                 fontsize=15, fontweight='bold')

    for idx, feat in enumerate(top_features):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col] if n_rows > 1 else axes[col]

        feat_idx = features.index(feat)
        feat_values = val_data_dict[h][feat].values
        feat_shap   = shap_dict[h][:, feat_idx]

        # Auto-pick the best interaction feature
        # (feature with highest interaction effect)
        shap.dependence_plot(
            feat_idx,
            shap_dict[h],
            val_data_dict[h],
            feature_names=features,
            ax=ax,
            show=False,
            dot_size=5,
            alpha=0.4
        )
        ax.set_title(feat, fontsize=10, fontweight='bold')

    # Hide unused subplots
    for idx in range(len(top_features), n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_path = f"{OUTPUT_DIR}/shap_dependence_h{h}.png"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")



5. SHAP DEPENDENCE PLOTS (Top 15 features per horizon)
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_dependence_h1.png
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_dependence_h3.png
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_dependence_h10.png
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_dependence_h25.png


In [8]:
# =====================================================================
# 6. COMBINED IMPORTANCE HEATMAP ACROSS HORIZONS
# =====================================================================
print("\n" + "=" * 70)
print("6. COMBINED IMPORTANCE HEATMAP")
print("=" * 70)

# Collect top features across all horizons
all_top_features = set()
for h in HORIZONS:
    all_top_features.update(importance_dfs[h].head(20)['feature'].tolist())
all_top_features = sorted(all_top_features)

# Build normalized importance matrix (rank within each horizon)
heatmap_data = pd.DataFrame(index=all_top_features, columns=HORIZONS, dtype=float)
for h in HORIZONS:
    imp = importance_dfs[h].set_index('feature')['mean_abs_shap']
    # Normalize to [0, 1] within each horizon for comparability
    imp_norm = imp / imp.max()
    for feat in all_top_features:
        heatmap_data.loc[feat, h] = imp_norm.get(feat, 0.0)

# Sort by average importance across horizons
heatmap_data['avg'] = heatmap_data.mean(axis=1)
heatmap_data = heatmap_data.sort_values('avg', ascending=False)
heatmap_data = heatmap_data.drop('avg', axis=1)

fig, ax = plt.subplots(figsize=(10, max(8, len(all_top_features) * 0.35)))
im = ax.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(HORIZONS)))
ax.set_xticklabels([f'H={h}' for h in HORIZONS], fontsize=11)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index, fontsize=8)

# Annotate cells
for i in range(len(heatmap_data)):
    for j in range(len(HORIZONS)):
        val = heatmap_data.values[i, j]
        color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color=color)

ax.set_title('Normalized Mean |SHAP| Heatmap (union of top-20 per horizon)',
             fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.6, label='Normalized importance')
plt.tight_layout()
save_path = f"{OUTPUT_DIR}/shap_heatmap_all_horizons.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Saved: {save_path}")



6. COMBINED IMPORTANCE HEATMAP
  Saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_heatmap_all_horizons.png


In [9]:
# =====================================================================
# 7. FEATURE SELECTION SUMMARY — DROP CANDIDATES
# =====================================================================
print("\n" + "=" * 70)
print("7. FEATURE SELECTION SUMMARY")
print("=" * 70)

# Build a summary DataFrame: mean |SHAP| per feature per horizon + rank
summary_rows = []
for feat in features:
    row = {'feature': feat}
    for h in HORIZONS:
        imp = importance_dfs[h].set_index('feature')
        row[f'shap_h{h}'] = imp.loc[feat, 'mean_abs_shap'] if feat in imp.index else 0.0
    # Average across horizons
    row['shap_avg'] = np.mean([row[f'shap_h{h}'] for h in HORIZONS])
    # Max across horizons (a feature might be critical for one horizon only)
    row['shap_max'] = np.max([row[f'shap_h{h}'] for h in HORIZONS])
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('shap_avg', ascending=False)

# Add rank columns
for h in HORIZONS:
    summary_df[f'rank_h{h}'] = summary_df[f'shap_h{h}'].rank(ascending=False).astype(int)
summary_df['rank_avg'] = summary_df['shap_avg'].rank(ascending=False).astype(int)

# Flag drop candidates: features ranked in bottom 30% for ALL horizons
n_features = len(features)
bottom_threshold = int(n_features * 0.7)  # rank > 70th percentile = bottom 30%
summary_df['drop_candidate'] = summary_df.apply(
    lambda r: all(r[f'rank_h{h}'] > bottom_threshold for h in HORIZONS),
    axis=1
)

save_path = f"{OUTPUT_DIR}/shap_feature_summary.csv"
summary_df.to_csv(save_path, index=False)
print(f"  Full summary saved: {save_path}")

# Print drop candidates
drop_df = summary_df[summary_df['drop_candidate']]
print(f"\n  DROP CANDIDATES ({len(drop_df)} features ranked in bottom 30% for ALL horizons):")
print(f"  {'feature':<20s} {'shap_avg':>10s} {'rank_h1':>8s} {'rank_h3':>8s} {'rank_h10':>9s} {'rank_h25':>9s}")
for _, r in drop_df.iterrows():
    print(f"  {r['feature']:<20s} {r['shap_avg']:>10.6f} "
          f"{int(r['rank_h1']):>8d} {int(r['rank_h3']):>8d} "
          f"{int(r['rank_h10']):>9d} {int(r['rank_h25']):>9d}")

# Print top-20 keepers
print(f"\n  TOP-20 FEATURES (by average SHAP across horizons):")
print(f"  {'feature':<20s} {'shap_avg':>10s} {'rank_h1':>8s} {'rank_h3':>8s} {'rank_h10':>9s} {'rank_h25':>9s}")
for _, r in summary_df.head(20).iterrows():
    print(f"  {r['feature']:<20s} {r['shap_avg']:>10.6f} "
          f"{int(r['rank_h1']):>8d} {int(r['rank_h3']):>8d} "
          f"{int(r['rank_h10']):>9d} {int(r['rank_h25']):>9d}")



7. FEATURE SELECTION SUMMARY
  Full summary saved: /home/lingod/kaggle_projects/ts_forecasting/shap_output/shap_feature_summary.csv

  DROP CANDIDATES (12 features ranked in bottom 30% for ALL horizons):
  feature                shap_avg  rank_h1  rank_h3  rank_h10  rank_h25
  feature_bc             0.000849       87       84        81        64
  feature_au             0.000778       71       74        75        67
  feature_bj             0.000639       69       67        74        78
  feature_bv             0.000532       70       81        71        77
  feature_ax             0.000480       81       87        77        70
  feature_aa             0.000467       74       69        85        80
  feature_c              0.000350       77       79        83        87
  feature_ba             0.000324       79       88        79        79
  feature_e              0.000291       80       83        86        82
  feature_bb             0.000253       86       86        84        81
  f

In [10]:
# =====================================================================
# 8. CORRELATED PAIR ANALYSIS (which one to drop?)
# =====================================================================
print("\n" + "=" * 70)
print("8. CORRELATED PAIR ANALYSIS — SHAP-BASED DROP RECOMMENDATION")
print("=" * 70)

# Known highly correlated pairs from EDA
corr_pairs = [
    ('feature_bz', 'feature_cd'),   # r=0.95
    ('feature_af', 'feature_u'),    # r=0.96
    ('feature_bo', 'feature_bm'),   # r=0.97
]

for f1, f2 in corr_pairs:
    shap1 = summary_df[summary_df['feature'] == f1]['shap_avg'].values
    shap2 = summary_df[summary_df['feature'] == f2]['shap_avg'].values
    if len(shap1) > 0 and len(shap2) > 0:
        shap1, shap2 = shap1[0], shap2[0]
        winner = f1 if shap1 >= shap2 else f2
        loser  = f2 if shap1 >= shap2 else f1
        print(f"  {f1} (SHAP={shap1:.6f}) vs {f2} (SHAP={shap2:.6f})")
        print(f"    → KEEP {winner}, consider dropping {loser}")
    else:
        print(f"  {f1} or {f2} not found in features (may be engineered)")



8. CORRELATED PAIR ANALYSIS — SHAP-BASED DROP RECOMMENDATION
  feature_bz (SHAP=0.146898) vs feature_cd (SHAP=0.017693)
    → KEEP feature_bz, consider dropping feature_cd
  feature_af (SHAP=0.048389) vs feature_u (SHAP=0.160171)
    → KEEP feature_u, consider dropping feature_af
  feature_bo (SHAP=0.002564) vs feature_bm (SHAP=0.004306)
    → KEEP feature_bm, consider dropping feature_bo


In [ ]:
# =====================================================================
print("\n" + "=" * 70)
print("DONE — All outputs saved to:", OUTPUT_DIR)
print("=" * 70)
print(f"""
Files produced:
  1. mean_abs_shap_per_horizon.png  — bar plots (global importance)
  2. shap_beeswarm_h{{1,3,10,25}}.png — beeswarm (value→direction)
  3. shap_dependence_h{{1,3,10,25}}.png — dependence (top {TOP_N_FEATURES} features)
  4. shap_heatmap_all_horizons.png  — cross-horizon comparison
  5. shap_feature_summary.csv       — full table with ranks & drop flags
""")
